In [ ]:
#分样本0.2
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
SEED = 12
set_seed(SEED)

# 加载数据


In [ ]:
exp=pd.read_csv('<RECIPE_PROJECT_ROOT>/data/sc11619genes422cell.csv')

meta=pd.read_csv('brforepridictmeta_dataall.csv')

rich_cells = meta[meta['fraction'] == 'Rich']['cell_names'].tolist()

# 由于 cell_names 在 exp 中是列名，需要检查格式是否匹配
rich_cells_exp = [cell for cell in rich_cells if cell in exp.columns]

# 提取 Rich 细胞对应的表达矩阵
exp_rich = exp[['Unnamed: 0'] + rich_cells_exp]




In [ ]:
exp_rich["scribo"] = exp_rich.iloc[:, 1:].mean(axis=1)


In [ ]:
# import pandas as pd

# # 读取 exp_rich 数据（如果是 DataFrame 变量，直接用 exp_rich）
# # exp_rich = pd.read_csv("your_file.csv")  # 如果是从 CSV 读取数据

# # 删除所有列均为 0 的行
# exp_rich_filtered = exp_rich.loc[~(exp_rich.iloc[:, 1:] == 0).all(axis=1)]

# # 显示结果
# exp_rich_filtered

# # 如果需要保存到文件
# # exp_rich_filtered.to_csv("filtered_exp_rich.csv", index=False)


In [ ]:
merged_df=pd.read_csv('<RECIPE_PROJECT_ROOT>/data/single_cell/scribo_reference.csv')

pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_rich_pause.csv')

# 规范化列名
pausing.columns = ['protein_id', "High_Pause_Countsscrich", "transcript_id"]

# 合并数据，缺失值填充为 0
merged_df2 = pd.merge(merged_df, pausing, on='transcript_id', how='left')
merged_df2['High_Pause_Countsscrich'].fillna(0, inplace=True)
pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_leu6h_pause.csv')

# 规范化列名
pausing.columns = ['protein_id', "High_Pause_Countsscleu6h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing', '_new'))
merged_df2['High_Pause_Countsscleu6h'].fillna(0, inplace=True)
pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_leu3h_pause.csv')
pausing.columns = ['protein_id', "High_Pause_Countsscleu3h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing', '_new'))
merged_df2['High_Pause_Countsscleu3h'].fillna(0, inplace=True)

pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_arg3h_pause.csv')

# 规范化列名
pausing.columns = ['protein_id', "High_Pause_Countsscarg3h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing1', '_new1'))
merged_df2['High_Pause_Countsscarg3h'].fillna(0, inplace=True)
pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_arg6h_pause.csv')
pausing.columns = ['protein_id', "High_Pause_Countsscarg6h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing2', '_new2'))
merged_df2['High_Pause_Countsscarg6h'].fillna(0, inplace=True)


In [ ]:
# import pandas as pd
# import re
# # 读取数据
# merged_df = pd.read_csv('./data/24077132kdncmergedf.csv')
# merged_df['protein'] = merged_df['protein_x'].str.split('.').str[0]

# pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_rich_pause.csv')

# # 规范化列名
# pausing.columns = ['protein_id', "High_Pause_Countssc", "transcript_id"]

# # 合并数据，缺失值填充为 0
# merged_df2 = pd.merge(merged_df, pausing, on='transcript_id', how='left')
# merged_df2['High_Pause_Countssc'].fillna(0, inplace=True)
# merged_df2['transcript_id'] = merged_df2['transcript_id'].apply(lambda x: re.sub(r'\.\d+$', '', x))

# # 筛选 `transcript_id` 交集
# common_transcripts = set(exp_rich_filtered['transcript_id']) & set(merged_df2['transcript_id'])

# # 选出 `merged_df2` 中 `transcript_id` 在交集中的行
# merged_df2_filtered = merged_df2[merged_df2['transcript_id'].isin(common_transcripts)]

# # 选出 `exp_rich_filtered` 中 `transcript_id` 在交集中的行
# exp_rich_filtered_final = exp_rich_filtered[exp_rich_filtered['transcript_id'].isin(common_transcripts)]

# exp_rich_filtered_final.shape


In [ ]:
exp_rich_filtered_final


In [ ]:
merged_df2_filtered


In [ ]:
# merged_df1 = pd.read_csv('Ribo_expyizhis0s12rich.csv')#改成count


# merged_df1["scribo"] = merged_df1.iloc[:, 1:].sum(axis=1)


In [ ]:
# exp=pd.read_csv('<PAUSING_SOURCE_ROOT>/data/scribo7132_422_normalized.csv')

# exp.head
# meta=pd.read_csv('brforepridictmeta_dataall.csv')

# rich_cells = meta[meta['fraction'] == 'Rich']['cell_names'].tolist()

# # 由于 cell_names 在 exp 中是列名，需要检查格式是否匹配
# rich_cells_exp = [cell for cell in rich_cells if cell in exp.columns]

# # 提取 Rich 细胞对应的表达矩阵
# exp_rich = exp[['transcript_id'] + rich_cells_exp]



In [ ]:
from torch_geometric.utils import from_scipy_sparse_matrix
ppi_matrix = pd.read_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix4p_pbulk11619.csv')

ppi_matrix.head()
all_sequence_outputsnew = np.load('./data/all_sequence_outputsnewbulk11619.npy')
ppi_matrix = sp.coo_matrix(ppi_matrix)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

In [ ]:

# merged_df = pd.read_csv('./data/24077132kdncmergedf.csv')
# merged_df['protein'] = merged_df['protein_x'].str.split('.').str[0]



# pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_scribo_pause.csv')

# pausing.columns = ['protein_id', "High_Pause_Countssc", "transcript_id"]
# merged_df2 = pd.merge(merged_df, pausing, on='transcript_id', how='left')
# merged_df2['High_Pause_Countssc'].fillna(0, inplace=True)


In [ ]:
# import pandas as pd
# from scipy import sparse as sp
# import pandas as pd
# import numpy as np
# import torch

# import torch.nn as nn

# ppi=pd.read_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corum4p_psc13000all.csv')

# #ppi= ppi.iloc[:,[1,7]]
# ppi.head


In [ ]:
merged_df2_filtered.shape

In [ ]:
# merged_df=merged_df2_filtered
# merged_df['protein'] = merged_df['protein'].str.split('.').str[0]
# merged_df.head()
# gene_id_dict = {gene: i for i, gene in enumerate(merged_df['protein'])} # 苏博的训练数据中的蛋白质id
# ppi['p1'] = ppi.protein_id1.map(gene_id_dict)
# ppi['p2'] = ppi.protein_id2.map(gene_id_dict)
# ppi_drop = ppi.dropna(subset=['p1','p2'])
# ppi_drop["CombinedScore"]=1
# ppi_matrix = sp.coo_matrix((ppi_drop["CombinedScore"], (ppi_drop['p1'], ppi_drop['p2'])), shape=(merged_df.shape[0], merged_df.shape[0]))
# ppi_matrix
# #保存ppi_matrix
# #sp.save_npz('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix.npz',ppi_matrix)

# ppi_matrix = ppi_matrix.toarray()
# ppi_matrix = pd.DataFrame(ppi_matrix)
# # ppi_matrix
# ppi_matrix.to_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix4p_p4431scribo.csv',index=False)

In [ ]:
cds_df=pd.read_csv("<PAUSING_SOURCE_ROOT>/cds_df38510.csv")
cds_df = cds_df.iloc[:,1:9]
cds_df['transcript_id'] = cds_df['transcript_id_x'].str.split('.').str[0]
exp=pd.read_csv("<PAUSING_SOURCE_ROOT>/data/sc11619genes422cell_normalized.csv")
# 将 cds_df 按照 exp 的基因列排序
sorted_cds_df = cds_df.set_index('transcript_id').reindex(exp['Unnamed: 0']).reset_index()

# 查看排序后的结果
sorted_cds_df.head()
sorted_cds_df.fillna(0, inplace=True)

merged_df=sorted_cds_df
merged_df['transcript_id'] = merged_df['transcript_id_x'].str.split('.').str[0]
merged_df.head

In [ ]:
merged_df3 = pd.read_csv('./data/24077132kdncmergedf.csv')
merged_df3['protein'] = merged_df3['protein_x'].str.split('.').str[0]
merged_df3.shape

In [ ]:

# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((exp_rich[['scribo']].values / np.median(exp_rich[['scribo']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df3[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
ppi_matrix = sp.coo_matrix(ppi_matrix)
# 转换为PyTorch Tensor
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = torch.tensor(merged_df2['High_Pause_Countsscrich'].values, dtype=torch.float32)
data.seq = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)


In [ ]:
paired_ratio = torch.tensor(np.array(merged_df2['High_Pause_Countsscrich'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

In [ ]:
print(ppi_matrix.shape)
print(merged_df.shape)



In [ ]:
exp_rich.head

In [ ]:
#分样本0.2
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
SEED = 8
set_seed(SEED)

# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((exp_rich[['scribo']].values / np.median(exp_rich[['scribo']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df3[['NC3']].values))+ 1)

y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)

# train_mask = (~torch.isnan(y)) & (y != 0)
paired_ratio = torch.tensor(np.array(merged_df2['High_Pause_Countsscrich'], dtype=np.float32).reshape(-1, 1))
# 创建图数据对象
# data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y,train_mask=train_mask)
# data.pause = paired_ratio
# data.seq = sequence_embedding


import torch

# 假设 y 是一个 torch.Tensor，形状为 (11619,)
valid_mask = (~torch.isnan(y)) & (y != 0)  # y 非 0 且非 NaN 的节点
valid_indices = valid_mask.nonzero(as_tuple=True)[0]  # 有效节点的索引
from sklearn.model_selection import train_test_split

# 先划分 train+val 和 test
train_val_idx, test_idx = train_test_split(valid_indices, test_size=0.2, random_state=42)

# 再从 train+val 中划分出 train 和 val
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2


num_nodes = y.shape[0]
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True
from torch_geometric.data import Data

data = Data(
    x=X,  # 节点特征: 11619 x N
    edge_index=edge_index,  # 图边结构
    y=y,  # 11619 x 1 或 11619,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)
data.pause = paired_ratio
data.seq = sequence_embedding

# # 筛选所有数据
# X_cpm_log2 = X_cpm_log2[valid_mask]
# y_cpm_log2 = y_cpm_log2[valid_mask]


# # # 数据准备
# y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
# X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# # 
# # y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# # X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))


# sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
# edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))
# sequence_embedding = sequence_embedding[valid_mask]
# paired_ratio = paired_ratio[valid_mask]


In [ ]:
len(X)

In [ ]:
exp_rich.shape

In [ ]:

# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
#SEED = 24
set_seed(SEED)

# # 数据集划分
# X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
#     X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

# X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
#     X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

# sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
#     sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

# sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
#     sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

# pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
#     paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

# pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
#     pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

# assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
# assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
# assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

# 训练与验证
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)

patience = 250 #试试更大的early stopping #300-0.495
num_epochs = 3000
patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 
for epoch in range(1, num_epochs + 1):
    # Training phase
    neural_net.train()
    optimizer.zero_grad()
    #out, z = neural_net(data)
    # train_loss = criterion(out[train_idx_X], data.y[train_idx_X]).mean()
    # train_loss.backward()
    # optimizer.step()
    out, z  = neural_net(data)
    train_loss = criterion(out[data.train_mask], data.y[data.train_mask])
    train_loss.backward()
    optimizer.step()

    train_r2 = r2_score(
        data.y[data.train_mask].cpu().numpy(),
        out[data.train_mask].detach().cpu().numpy()
    )
    # Validation phase
    neural_net.eval()
    with torch.no_grad():
        val_out, _ = neural_net(data)
        val_loss = criterion(val_out[data.val_mask], data.y[data.val_mask]).mean()
        val_r2 = r2_score(data.y[data.val_mask].cpu().numpy(), val_out[data.val_mask].cpu().numpy())

        test_out, _ = neural_net(data)
        test_loss = criterion(test_out[data.test_mask], data.y[data.test_mask]).mean()
        test_r2 = r2_score(data.y[data.test_mask].cpu().numpy(), test_out[data.test_mask].cpu().numpy())

    # Check if validation loss has improved
    if val_r2 > best_val_r2:  # 使用 R² 作为早停指标
        best_val_r2 = val_r2
        best_train_loss = train_loss
        best_val_loss = val_loss
        patience_counter = 0

        best_test_loss = test_loss
        best_test_r2 = test_r2
        torch.save(neural_net.state_dict(), './models/single_cell_module_a_seed8_candidate.pt')
        y_true_np = data.y[data.test_mask].cpu().detach().numpy()
        y_pred_np = test_out[data.test_mask].cpu().detach().numpy()


    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Train R²: {train_r2:.3f}| "
          f"Val Loss: {best_val_loss:.3f}, Val R²: {best_val_r2:.3f}|"
          f"Test Loss: {best_test_loss:.3f}, Test R²: {best_test_r2:.3f}")

微调部分

In [ ]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
model = NeuralGraph().to(device)

# ✅ 加载预训练模型
pretrained_model_path = './models/bulk_self_learning_pretrained.pt'
if os.path.exists(pretrained_model_path):
    model.load_state_dict(torch.load(pretrained_model_path, map_location=device))
    print("✅ 加载预训练模型成功！")
else:
    print("⚠️ 预训练模型未找到，将从头训练！")

# ✅ 加载你的数据
data = data.to(device)

# ✅ 指定训练/验证/测试索引或掩码（任选其一）
train_idx = data.train_mask
val_idx = data.val_mask
test_idx = data.test_mask

# ✅ 训练参数
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
num_epochs = 1000
patience = 100
patience_counter = 0
best_val_r2 = float('-inf')
best_model_path = './models/single_cell_module_a_seed8_best.pt'

# ✅ 训练循环
for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()
    out, _ = model(data)
    loss = criterion(out[train_idx], data.y[train_idx])
    loss.backward()
    optimizer.step()

    train_r2 = r2_score(
        data.y[train_idx].cpu().numpy(),
        out[train_idx].detach().cpu().numpy()
    )

    # 验证
    model.eval()
    with torch.no_grad():
        out_val, _ = model(data)
        val_loss = criterion(out_val[val_idx], data.y[val_idx])
        val_r2 = r2_score(
            data.y[val_idx].cpu().numpy(),
            out_val[val_idx].cpu().numpy()
        )
        test_out, _ = model(data)
        test_loss = criterion(test_out[test_idx], data.y[test_idx])
        test_r2 = r2_score(
            data.y[test_idx].cpu().numpy(),
            test_out[test_idx].cpu().numpy()
        )

    # Early stopping
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_train_loss = loss.item()
        best_val_loss = val_loss.item()
        best_test_loss = test_loss.item()
        best_test_r2 = test_r2
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"[Epoch {epoch}] ✅ New best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"⏹ Early stopping at epoch {epoch}")
            break

    print(f"Epoch {epoch:03d} | Train Loss: {loss:.3f}, Train R²: {train_r2:.3f} | "
          f"Val Loss: {val_loss:.3f}, Val R²: {val_r2:.3f} | "
          f"Test Loss: {test_loss:.3f}, Test R²: {test_r2:.3f}")

print("✅ 微调完成，最佳模型已保存！")


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色设置
scatter_color = '#C7B0C4'  # 散点图的颜色
hist_color = '#BFD2DF'  # 直方图的颜色

# ✅ 只取4258个有标签的蛋白
# 假设你的全体预测是 y_pred，真实值是 y_real，labeled_idx 是4258的索引
# 修改1️⃣：只取 labeled_idx 部分
original_y_np = y[valid_indices].cpu().numpy().flatten()      # 真实值
y_all_pred_np = model(data)[0][valid_indices].detach().cpu().numpy().flatten()  # 预测值

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]  # 计算Pearson相关系数
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')  # 使用pingouin计算p值
p_value = result['p-val'][0]  # 提取p值

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"  # 若p值小于非常小的数值，格式化为科学计数法
else:
    p_value_sci = f"{p_value:.2e}"  # 格式化为科学计数法
    p_base, p_exp = p_value_sci.split("e")  # 分割科学计数法的基数和指数
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"  # LaTeX格式化p值

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))  # 设置画布大小
gs = GridSpec(4, 4)  # 创建网格布局

# 定义各个子图的区域
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 散点图区域
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # x轴直方图区域
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # y轴直方图区域

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)  # 绘制散点图

# ➕ 对角线参考线 (表示完全一致的情况)
max_val = max(original_y_np.max(), y_all_pred_np.max())  # 获取最大值以确定对角线的范围
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')  # 绘制对角线

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)  # 绘制回归线

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)  # 设置x轴标签
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)  # 设置y轴标签
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)  # 添加相关系数文本
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')  # 添加p值文本

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)  # 绘制x轴直方图
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)  # 绘制y轴直方图

ax_histy.yaxis.set_visible(False)  # 隐藏y轴
ax_histx.set_ylabel("Frequency")  # 设置x轴直方图的y轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏x轴

# 💾 保存图形
plt.savefig("./outputs/figures/single_cell_module_a_prediction_vs_observed.pdf", format="pdf", bbox_inches="tight")

# 📈 展示图形
plt.show()


In [ ]:
# 获取训练集、验证集和测试集的索引
train_idx = data.train_mask
val_idx = data.val_mask
test_idx = data.test_mask

# 检查是否有重合
train_val_overlap = torch.sum(train_idx & val_idx).item()  # 训练集和验证集重合的数量
train_test_overlap = torch.sum(train_idx & test_idx).item()  # 训练集和测试集重合的数量
val_test_overlap = torch.sum(val_idx & test_idx).item()  # 验证集和测试集重合的数量

# 输出重合的结果
print(f"Train and Val overlap: {train_val_overlap}")
print(f"Train and Test overlap: {train_test_overlap}")
print(f"Val and Test overlap: {val_test_overlap}")

# 检查是否有任何交集
if train_val_overlap > 0 or train_test_overlap > 0 or val_test_overlap > 0:
    print("There is overlap between train, val, and test sets.")
else:
    print("No overlap between train, val, and test sets.")


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色设置
scatter_color = '#C7B0C4'  # 散点图的颜色
hist_color = '#BFD2DF'  # 直方图的颜色
val_idx = val_idx.cpu()

# ✅ 只取4258个有标签的蛋白
# 假设你的全体预测是 y_pred，真实值是 y_real，labeled_idx 是4258的索引
# 修改1️⃣：只取 labeled_idx 部分
original_y_np = y[val_idx].cpu().numpy().flatten()      # 真实值
y_all_pred_np = model(data)[0][val_idx].detach().cpu().numpy().flatten()  # 预测值

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]  # 计算Pearson相关系数
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')  # 使用pingouin计算p值
p_value = result['p-val'][0]  # 提取p值

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"  # 若p值小于非常小的数值，格式化为科学计数法
else:
    p_value_sci = f"{p_value:.2e}"  # 格式化为科学计数法
    p_base, p_exp = p_value_sci.split("e")  # 分割科学计数法的基数和指数
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"  # LaTeX格式化p值

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))  # 设置画布大小
gs = GridSpec(4, 4)  # 创建网格布局

# 定义各个子图的区域
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 散点图区域
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # x轴直方图区域
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # y轴直方图区域

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)  # 绘制散点图

# ➕ 对角线参考线 (表示完全一致的情况)
max_val = max(original_y_np.max(), y_all_pred_np.max())  # 获取最大值以确定对角线的范围
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')  # 绘制对角线

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)  # 绘制回归线

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)  # 设置x轴标签
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)  # 设置y轴标签
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)  # 添加相关系数文本
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')  # 添加p值文本

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)  # 绘制x轴直方图
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)  # 绘制y轴直方图

ax_histy.yaxis.set_visible(False)  # 隐藏y轴
ax_histx.set_ylabel("Frequency")  # 设置x轴直方图的y轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏x轴

# 💾 保存图形
plt.savefig("./outputs/figures/single_cell_module_a_validation.pdf", format="pdf", bbox_inches="tight")

# 📈 展示图形
plt.show()


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色设置
scatter_color = '#C7B0C4'  # 散点图的颜色
hist_color = '#BFD2DF'  # 直方图的颜色
test_idx = test_idx.cpu()

# ✅ 只取4258个有标签的蛋白
# 假设你的全体预测是 y_pred，真实值是 y_real，labeled_idx 是4258的索引
# 修改1️⃣：只取 labeled_idx 部分
original_y_np = y[test_idx].cpu().numpy().flatten()      # 真实值
y_all_pred_np = model(data)[0][test_idx].detach().cpu().numpy().flatten()  # 预测值

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]  # 计算Pearson相关系数
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')  # 使用pingouin计算p值
p_value = result['p-val'][0]  # 提取p值

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"  # 若p值小于非常小的数值，格式化为科学计数法
else:
    p_value_sci = f"{p_value:.2e}"  # 格式化为科学计数法
    p_base, p_exp = p_value_sci.split("e")  # 分割科学计数法的基数和指数
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"  # LaTeX格式化p值

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))  # 设置画布大小
gs = GridSpec(4, 4)  # 创建网格布局

# 定义各个子图的区域
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 散点图区域
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # x轴直方图区域
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # y轴直方图区域

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)  # 绘制散点图

# ➕ 对角线参考线 (表示完全一致的情况)
max_val = max(original_y_np.max(), y_all_pred_np.max())  # 获取最大值以确定对角线的范围
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')  # 绘制对角线

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)  # 绘制回归线

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)  # 设置x轴标签
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)  # 设置y轴标签
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)  # 添加相关系数文本
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')  # 添加p值文本

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)  # 绘制x轴直方图
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)  # 绘制y轴直方图

ax_histy.yaxis.set_visible(False)  # 隐藏y轴
ax_histx.set_ylabel("Frequency")  # 设置x轴直方图的y轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏x轴

# 💾 保存图形
plt.savefig("./outputs/figures/single_cell_module_a_test.pdf", format="pdf", bbox_inches="tight")

# 📈 展示图形
plt.show()


## 出图

In [ ]:
# # 运行到这之后
# num_nodes = y.shape[0]
# train_mask = torch.zeros(num_nodes, dtype=torch.bool)
# val_mask = torch.zeros(num_nodes, dtype=torch.bool)
# test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# train_mask[train_idx] = True
# val_mask[val_idx] = True
# test_mask[test_idx] = True
# from torch_geometric.data import Data

# data = Data(
#     x=X,  # 节点特征: 11619 x N
#     edge_index=edge_index,  # 图边结构
#     y=y,  # 11619 x 1 或 11619,
#     train_mask=train_mask,
#     val_mask=val_mask,
#     test_mask=test_mask
# )
# data.pause = paired_ratio
# data.seq = sequence_embedding

In [ ]:

# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
#SEED = 24
set_seed(SEED)

# # 数据集划分
# X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
#     X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

# X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
#     X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

# sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
#     sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

# sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
#     sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

# pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
#     paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

# pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
#     pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

# assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
# assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
# assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

# 训练与验证
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)

patience = 250 #试试更大的early stopping #300-0.495
num_epochs = 3000
patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 

In [ ]:
# ✅ 指定训练/验证/测试索引或掩码（任选其一）
train_idx = data.train_mask
val_idx = data.val_mask
test_idx = data.test_mask

# ✅ 训练参数
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
num_epochs = 1000
patience = 100
patience_counter = 0
best_val_r2 = float('-inf')
best_model_path = './models/single_cell_module_a_seed8_best.pt'

In [ ]:

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print("✅ 加载预训练模型成功！")
else:
    print("⚠️ 预训练模型未找到，将从头训练！")
out, _ = model(data)
#loss = criterion(out[train_idx], data.y[train_idx])

In [ ]:
import torch

# 假设 y 是一个 torch.Tensor，形状为 (11619,)
valid_mask = (~torch.isnan(y)) & (y != 0)  # y 非 0 且非 NaN 的节点
valid_indices = valid_mask.nonzero(as_tuple=True)[0]  # 有效节点的索引
from sklearn.model_selection import train_test_split

# 先划分 train+val 和 test
train_val_idx, test_idx = train_test_split(valid_indices, test_size=0.2, random_state=42)

# 再从 train+val 中划分出 train 和 val
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2


num_nodes = y.shape[0]
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

In [ ]:
len(train_val_idx)

In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色设置
scatter_color = '#C7B0C4'  # 散点图的颜色
hist_color = '#BFD2DF'  # 直方图的颜色

# ✅ 只取4258个有标签的蛋白
# 假设你的全体预测是 y_pred，真实值是 y_real，labeled_idx 是4258的索引
# 修改1️⃣：只取 labeled_idx 部分
original_y_np = y[valid_indices].cpu().numpy().flatten()      # 真实值
y_all_pred_np = out[valid_indices].detach().cpu().numpy().flatten()  # 预测值

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]  # 计算Pearson相关系数
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')  # 使用pingouin计算p值
p_value = result['p-val'][0]  # 提取p值

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"  # 若p值小于非常小的数值，格式化为科学计数法
else:
    p_value_sci = f"{p_value:.2e}"  # 格式化为科学计数法
    p_base, p_exp = p_value_sci.split("e")  # 分割科学计数法的基数和指数
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"  # LaTeX格式化p值

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))  # 设置画布大小
gs = GridSpec(4, 4)  # 创建网格布局

# 定义各个子图的区域
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 散点图区域
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # x轴直方图区域
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # y轴直方图区域

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)  # 绘制散点图

# ➕ 对角线参考线 (表示完全一致的情况)
max_val = max(original_y_np.max(), y_all_pred_np.max())  # 获取最大值以确定对角线的范围
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')  # 绘制对角线

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)  # 绘制回归线

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)  # 设置x轴标签
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)  # 设置y轴标签
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)  # 添加相关系数文本
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')  # 添加p值文本

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)  # 绘制x轴直方图
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)  # 绘制y轴直方图

ax_histy.yaxis.set_visible(False)  # 隐藏y轴
ax_histx.set_ylabel("Frequency")  # 设置x轴直方图的y轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏x轴

# 💾 保存图形
#plt.savefig("./outputs/figures/single_cell_module_a_all_labeled.pdf", format="pdf", bbox_inches="tight")

# 📈 展示图形
plt.show()


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色设置
scatter_color = '#C7B0C4'  # 散点图的颜色
hist_color = '#BFD2DF'  # 直方图的颜色

# ✅ 只取4258个有标签的蛋白
# 假设你的全体预测是 y_pred，真实值是 y_real，labeled_idx 是4258的索引
# 修改1️⃣：只取 labeled_idx 部分
original_y_np = y[train_val_idx].cpu().numpy().flatten()      # 真实值
y_all_pred_np = model(data)[0][train_val_idx].detach().cpu().numpy().flatten()  # 预测值

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]  # 计算Pearson相关系数
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')  # 使用pingouin计算p值
p_value = result['p-val'][0]  # 提取p值

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"  # 若p值小于非常小的数值，格式化为科学计数法
else:
    p_value_sci = f"{p_value:.2e}"  # 格式化为科学计数法
    p_base, p_exp = p_value_sci.split("e")  # 分割科学计数法的基数和指数
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"  # LaTeX格式化p值

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))  # 设置画布大小
gs = GridSpec(4, 4)  # 创建网格布局

# 定义各个子图的区域
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 散点图区域
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # x轴直方图区域
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # y轴直方图区域

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)  # 绘制散点图

# ➕ 对角线参考线 (表示完全一致的情况)
max_val = max(original_y_np.max(), y_all_pred_np.max())  # 获取最大值以确定对角线的范围
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')  # 绘制对角线

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)  # 绘制回归线

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)  # 设置x轴标签
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)  # 设置y轴标签
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)  # 添加相关系数文本
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')  # 添加p值文本

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)  # 绘制x轴直方图
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)  # 绘制y轴直方图

ax_histy.yaxis.set_visible(False)  # 隐藏y轴
ax_histx.set_ylabel("Frequency")  # 设置x轴直方图的y轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏x轴

# 💾 保存图形
plt.savefig("./outputs/figures/single_cell_module_a_train.pdf", format="pdf", bbox_inches="tight")

# 📈 展示图形
plt.show()


In [ ]:

# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
SEED = 12
set_seed(SEED)

# 数据集划分
X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
    X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
    X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
    sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
    sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
    paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
    pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

In [ ]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
model = NeuralGraph().to(device)

# **加载预训练模型**
pretrained_model_path = './models/bulk_unknown_seed12_best.pt'
if os.path.exists(pretrained_model_path):
    model.load_state_dict(torch.load(pretrained_model_path))
    print("加载预训练模型成功！")
else:
    print("预训练模型未找到，将从头训练！")

# **3. 加载新的数据**
# 请替换您的数据路径
data = data.to(device)
# **4. 进行数据集划分**

# **5. 设置优化器和损失函数**
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # 微调学习率
criterion = nn.MSELoss()
num_epochs = 1000
patience = 100
best_val_loss = float('inf')
patience_counter = 0

patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 

for epoch in range(1, num_epochs + 1):
    # Training phase
    model.train()
    optimizer.zero_grad()
    out, z = model(data)
    train_loss = criterion(out[train_idx_X], data.y[train_idx_X]).mean()
    train_loss.backward()
    optimizer.step()

    train_r2 = r2_score(
        data.y[train_idx_X].cpu().numpy(),
        out[train_idx_X].detach().cpu().numpy()
    )

    # Validation phase
    model.eval()
    with torch.no_grad():
        val_out, _ = model(data)
        val_loss = criterion(val_out[val_idx_X], data.y[val_idx_X]).mean()
        val_r2 = r2_score(data.y[val_idx_X].cpu().numpy(), val_out[val_idx_X].cpu().numpy())

        test_out, _ = model(data)
        test_loss = criterion(test_out[test_idx_X], data.y[test_idx_X]).mean()
        test_r2 = r2_score(data.y[test_idx_X].cpu().numpy(), test_out[test_idx_X].cpu().numpy())

    # Check if validation loss has improved
    if val_r2 > best_val_r2:  # 使用 R² 作为早停指标
        best_val_r2 = val_r2
        best_train_loss = train_loss
        best_val_loss = val_loss
        patience_counter = 0

        best_test_loss = test_loss
        best_test_r2 = test_r2
        torch.save(model.state_dict(), './models/single_cell_module_a_finetuned.pt')
        y_true_np = data.y[test_idx_X].cpu().detach().numpy()
        y_pred_np = test_out[test_idx_X].cpu().detach().numpy()
        print(f"Epoch {epoch}: New best model saved!")

    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Train R²: {train_r2:.3f}| "
          f"Val Loss: {best_val_loss:.3f}, Val R²: {best_val_r2:.3f}| "
          f"Test Loss: {best_test_loss:.3f}, Test R²: {best_test_r2:.3f}")

print("微调完成，最佳模型已保存！")